# BERT Token Classification for Term Extraction

This notebook demonstrates a BERT-based approach to term extraction:
- Uses BIO tagging scheme (Beginning-Inside-Outside)
- Fine-tunes BERT model for token classification
- Trains on labeled data to recognize term boundaries

## Setup and Imports

In [19]:
import random
import json
import os
import numpy as np
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorForTokenClassification
)
#from utils import seed_everything, get_device, move_to_device, print_device_info, maybe_empty_cache
from torch.utils.data import Dataset
import pandas as pd

In [20]:
###############################
# Device Management Functions #
###############################

def _backend_available(device_type: str) -> bool:
    """Check backend availability, including on older PyTorch builds."""
    if device_type == "cpu":
        return True

    if device_type == "mps":
        backend = getattr(torch.backends, "mps", None)
    elif device_type in ("cuda", "xpu"):
        backend = getattr(torch, device_type, None)
    else:
        return False

    return backend is not None and backend.is_available()


def get_device(preferred=None) -> torch.device:
    """
    Select an available device.

    With no preference, preserve the original priority and add XPU:
        MPS -> CUDA -> XPU -> CPU

    Examples:
        get_device()         # Automatic selection
        get_device("xpu")    # Require an Intel GPU
        get_device("xpu:0")  # Require Intel GPU 0
        get_device("cuda:0")
        get_device("cpu")

    An explicitly requested unavailable device raises an error.
    """
    if preferred is None:
        for device_type in ("mps", "cuda", "xpu", "cpu"):
            if _backend_available(device_type):
                return torch.device(device_type)

    device = torch.device(preferred)

    if device.type not in ("cpu", "cuda", "xpu", "mps"):
        raise ValueError(f"Unsupported device type: {device.type}")

    if not _backend_available(device.type):
        raise RuntimeError(
            f"Requested device '{device}' is unavailable. "
            "Check your PyTorch build, GPU drivers, and hardware."
        )

    if device.type in ("cuda", "xpu") and device.index is not None:
        backend = getattr(torch, device.type)
        device_count = backend.device_count()

        if not 0 <= device.index < device_count:
            raise ValueError(
                f"Invalid device index {device.index}: "
                f"{device_count} {device.type} device(s) available."
            )

    return device


def move_to_device(data, device: torch.device, non_blocking=False):
    """
    Move tensors or models to a device, including nested containers.

    Non-tensor values such as strings, numbers, and None are preserved.
    """
    device = torch.device(device)

    if isinstance(data, torch.Tensor):
        return data.to(device=device, non_blocking=non_blocking)

    if isinstance(data, torch.nn.Module):
        return data.to(device=device, non_blocking=non_blocking)

    if isinstance(data, dict):
        return {
            key: move_to_device(value, device, non_blocking)
            for key, value in data.items()
        }

    if isinstance(data, list):
        return [
            move_to_device(item, device, non_blocking)
            for item in data
        ]

    if isinstance(data, tuple):
        items = [
            move_to_device(item, device, non_blocking)
            for item in data
        ]

        # Preserve named tuples.
        if hasattr(data, "_fields"):
            return type(data)(*items)

        return tuple(items)

    return data


def print_device_info(device: torch.device) -> None:
    """Print information about the selected CPU or GPU."""
    device = get_device(device)
    print(f"Device: {device}")

    if device.type in ("cuda", "xpu"):
        backend = getattr(torch, device.type)
        index = (
            device.index
            if device.index is not None
            else backend.current_device()
        )
        properties = backend.get_device_properties(index)
        gib = 1024 ** 3

        print(f"Backend: {device.type.upper()}")
        print(f"Device index: {index}")
        print(f"GPU name: {backend.get_device_name(index)}")
        print(f"GPU count: {backend.device_count()}")
        print(f"Total memory: {properties.total_memory / gib:.2f} GiB")
        print(
            f"Allocated memory: "
            f"{backend.memory_allocated(index) / gib:.2f} GiB"
        )
        print(
            f"Reserved memory: "
            f"{backend.memory_reserved(index) / gib:.2f} GiB"
        )

    elif device.type == "mps":
        print("MPS backend is active")

        memory_functions = (
            ("Recommended max memory", "recommended_max_memory"),
            ("Current allocated memory", "current_allocated_memory"),
            ("Driver allocated memory", "driver_allocated_memory"),
        )

        for label, function_name in memory_functions:
            function = getattr(torch.mps, function_name, None)

            if callable(function):
                print(f"{label}: {function() / (1024 ** 3):.2f} GiB")

    else:
        print("CPU backend is active")


def maybe_empty_cache(device: torch.device) -> None:
    """
    Release unused GPU allocator cache.

    This does not free memory occupied by live tensors or models.
    """
    device = torch.device(device)

    if not _backend_available(device.type):
        return

    if device.type in ("cuda", "xpu"):
        backend = getattr(torch, device.type)

        # Respect the requested GPU index.
        with backend.device(device):
            backend.empty_cache()

    elif device.type == "mps":
        empty_cache = getattr(torch.mps, "empty_cache", None)

        if callable(empty_cache):
            empty_cache()


#####################
# Seeding Functions #
#####################

def seed_everything(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch random number generators."""
    random.seed(seed)
    np.random.seed(seed)

    # PyTorch seeds its CPU and supported accelerator generators.
    torch.manual_seed(seed)

In [21]:
# Set random seeds for reproducibility
seed_everything(42)

print(f"PyTorch version: {torch.__version__}")
device = get_device()
maybe_empty_cache(device)
print_device_info(device)

PyTorch version: 2.14.1+xpu
Device: xpu
Backend: XPU
Device index: 0
GPU name: Intel(R) Arc(TM) 140V GPU (16GB)
GPU count: 1
Total memory: 16.48 GiB
Allocated memory: 0.61 GiB
Reserved memory: 0.63 GiB


In [ ]:
# Define label mappings for BIO tagging scheme
label_list = ['O', 'B-TERM', 'I-TERM']
label2id = {k: v for v, k in enumerate(label_list)}
id2label = {v: k for v, k in enumerate(label_list)}

print(f"Labels: {label_list}")
print(f"Label to ID: {label2id}")
print(f"ID to Label: {id2label}")

# Model configuration
model_name = "google-bert/bert-base-uncased"
#model_name = "google-bert/bert-base-cased"
output_model_dir = "models/" + (model_name.split("/")[-1]).lower()

print(f"\nModel: {model_name}")
print(f"Output directory: {output_model_dir}")

Labels: ['O', 'B-TERM', 'I-TERM']
Label to ID: {'O': 0, 'B-TERM': 1, 'I-TERM': 2}
ID to Label: {0: 'O', 1: 'B-TERM', 2: 'I-TERM'}

Model: google-bert/bert-base-uncased
Output directory: models/bert-base-uncased


## Data Loading and Processing

In [23]:
# Load training and dev data
train_data = json.load(open('../data/taskA/train.json', 'r', encoding='utf-8'))

document_ids = sorted(list(train_data.keys()))
random.shuffle(document_ids) # we seeded above for reproducibility

dev_size = round(len(document_ids) * 0.2)  # Use 20% of the data for dev
dev_ids = document_ids[:dev_size]
train_ids = document_ids[dev_size:]

dev_data = {doc_id: train_data[doc_id] for doc_id in dev_ids}
train_data_tmp = {doc_id: train_data[doc_id] for doc_id in train_ids}
train_data = train_data_tmp  # Rename for clarity

print(f"Training documents: {len(train_data)}")
print(f"Dev documents: {len(dev_data)}")
print(f"\nExample document:")
for id, content in train_data.items():
    print(f"Document ID: {id}")
    print(f"  Text: {content['text']}")
    print(f"  Terms: {content['terms']}")
    break  

Training documents: 432
Dev documents: 108

Example document:
Document ID: mh_87
  Text: Microbiota from Alzheimer's patients induce deficits in cognition and hippocampal neurogenesis.  Alzheimer's disease is a complex neurodegenerative disorder leading to a decline in cognitive function and mental health. Recent research has positioned the gut microbiota as an important susceptibility factor in Alzheimer's disease by showing specific alterations in the gut microbiome composition of Alzheimer's patients and in rodent models. However, it is unknown whether gut microbiota alterations are causal in the manifestation of Alzheimer's symptoms. To understand the involvement of Alzheimer's patient gut microbiota in host physiology and behaviour, we transplanted faecal microbiota from Alzheimer's patients and age-matched healthy controls into microbiota-depleted young adult rats. We found impairments in behaviours reliant on adult hippocampal neurogenesis, an essential process for certain memor

## Evaluation Metrics

Using the official evaluation metrics from the competition.

In [24]:
def micro_f1_score(gold_standard, system_output):
    """
    Evaluates performance using Precision, Recall, and F1 score 
    based on individual term matching (micro-average).
    """
    total_true_positives = 0
    total_false_positives = 0
    total_false_negatives = 0
    
    for gold, system in zip(gold_standard, system_output):
        gold_set = set(gold)
        system_set = set(system)
        
        true_positives = len(gold_set.intersection(system_set))
        false_positives = len(system_set - gold_set)
        false_negatives = len(gold_set - system_set)
        
        total_true_positives += true_positives
        total_false_positives += false_positives
        total_false_negatives += false_negatives
    
    precision = total_true_positives / (total_true_positives + total_false_positives) if (total_true_positives + total_false_positives) > 0 else 0
    recall = total_true_positives / (total_true_positives + total_false_negatives) if (total_true_positives + total_false_negatives) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1, total_true_positives, total_false_positives, total_false_negatives


def type_f1_score(gold_standard, system_output):
    """
    Evaluates performance using Type Precision, Type Recall, and Type F1 score
    based on the set of unique terms extracted at least once across the entire dataset.
    """
    all_gold_terms = set()
    for item_terms in gold_standard:
        all_gold_terms.update(item_terms)
    
    all_system_terms = set()
    for item_terms in system_output:
        all_system_terms.update(item_terms)
    
    type_true_positives = len(all_gold_terms.intersection(all_system_terms))
    type_false_positives = len(all_system_terms - all_gold_terms)
    type_false_negatives = len(all_gold_terms - all_system_terms)
    
    type_precision = type_true_positives / (type_true_positives + type_false_positives) if (type_true_positives + type_false_positives) > 0 else 0
    type_recall = type_true_positives / (type_true_positives + type_false_negatives) if (type_true_positives + type_false_negatives) > 0 else 0
    type_f1 = 2 * (type_precision * type_recall) / (type_precision + type_recall) if (type_precision + type_recall) > 0 else 0
    
    return type_precision, type_recall, type_f1


print("✓ Evaluation functions defined")

✓ Evaluation functions defined


## Initialize BERT Model and Tokenizer

In [25]:
# Initialize tokenizer and model
print("Initializing BERT tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name, 
    num_labels=len(label_list), 
    id2label=id2label, 
    label2id=label2id
)

print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"Model loaded with {model.num_labels} labels")
print(f"Vocabulary size: {tokenizer.vocab_size}")

# Test tokenization
sample_text = "the brown fox jumps over the lazy dog"
tokens = tokenizer.tokenize(sample_text)
print(f"\nSample tokenization: {tokens}")

Initializing BERT tokenizer and model...


Some weights of BertForTokenClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizer loaded: BertTokenizerFast
Model loaded with 3 labels
Vocabulary size: 30522

Sample tokenization: ['the', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']


## BIO Tag Generation for Training Data

In [26]:
def create_ner_tags(text, terms, tokenizer, label2id):
    """
    Create NER tags for tokenized text based on the given terms.
    Uses BIO scheme: B-TERM for beginning, I-TERM for inside, O for outside.
    """
    tokens = tokenizer.tokenize(text)
    ner_tags = ['O'] * len(tokens)
    
    # Sort terms by length (descending) to handle overlapping terms
    sorted_terms = sorted(terms, key=len, reverse=True)
    
    for term in sorted_terms:
        normalized_text = text.lower()
        normalized_term = term.lower()
        
        # Find all occurrences of the term
        term_positions = []
        start = 0
        while True:
            pos = normalized_text.find(normalized_term, start)
            if pos == -1:
                break
            term_positions.append((pos, pos + len(normalized_term)))
            start = pos + 1
        
        # Map character positions to token positions
        for start_char, end_char in term_positions:
            token_start_idx = None
            token_end_idx = None
            
            encoding = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
            offsets = encoding['offset_mapping']
            
            for i, (token_start, token_end) in enumerate(offsets):
                if token_start < end_char and token_end > start_char:
                    if token_start_idx is None:
                        token_start_idx = i
                    token_end_idx = i
            
            # Apply BIO tagging
            if token_start_idx is not None and token_end_idx is not None:
                for i in range(token_start_idx, token_end_idx + 1):
                    if i < len(ner_tags) and ner_tags[i] == 'O':
                        if i == token_start_idx:
                            ner_tags[i] = 'B-TERM'
                        else:
                            ner_tags[i] = 'I-TERM'
    
    ner_tag_ids = [label2id[tag] for tag in ner_tags]
    return tokens, ner_tag_ids


print("BIO tag generation function defined")

BIO tag generation function defined


## Process Training and Dev Data with BIO Tags

In [27]:
import pandas as pd
from tqdm import tqdm

# Process training data
for id, entry in tqdm(train_data.items(), desc="Processing training data..."):
    text = entry['text']
    terms = entry['terms']
    
    tokens, ner_tags = create_ner_tags(text, terms, tokenizer, label2id)
    entry['tokens'] = tokens
    entry['ner_tags'] = ner_tags
    
print(f"Training data processed: {len(train_data)} documents")


print(f"\nSample train sentence:")
for id, content in train_data.items():
    print(f"Document ID: {id}")
    print(f"  Text: {content['text']}")
    print(f"  Terms: {content['terms']}")
    break
    token_tags = []
    for token, tag in zip(content['tokens'], content['ner_tags']):
        token_tags.append((token, id2label[tag]))
    print(f"\n{pd.DataFrame(token_tags, columns=['Token', 'Tag']).to_markdown()}")

Processing training data...: 100%|██████████| 432/432 [00:54<00:00,  7.97it/s]

Training data processed: 432 documents

Sample train sentence:
Document ID: mh_87
  Text: Microbiota from Alzheimer's patients induce deficits in cognition and hippocampal neurogenesis.  Alzheimer's disease is a complex neurodegenerative disorder leading to a decline in cognitive function and mental health. Recent research has positioned the gut microbiota as an important susceptibility factor in Alzheimer's disease by showing specific alterations in the gut microbiome composition of Alzheimer's patients and in rodent models. However, it is unknown whether gut microbiota alterations are causal in the manifestation of Alzheimer's symptoms. To understand the involvement of Alzheimer's patient gut microbiota in host physiology and behaviour, we transplanted faecal microbiota from Alzheimer's patients and age-matched healthy controls into microbiota-depleted young adult rats. We found impairments in behaviours reliant on adult hippocampal neurogenesis, an essential process for certain memo

In [28]:
# Process dev data
print("\nProcessing dev data...")
for id, entry in tqdm(dev_data.items(), desc="Processing dev data..."):
    text = entry['text']
    terms = entry['terms']
    
    tokens, ner_tags = create_ner_tags(text, terms, tokenizer, label2id)
    entry['tokens'] = tokens
    entry['ner_tags'] = ner_tags
    
print(f"Dev data processed: {len(dev_data)} sentences")

print(f"\nSample dev sentence:")
for id, content in dev_data.items():
    print(f"Document ID: {id}")
    print(f"  Text: {content['text']}")
    print(f"  Terms: {content['terms']}")
    break
    token_tags = []
    for token, tag in zip(content['tokens'], content['ner_tags']):
        token_tags.append((token, id2label[tag]))
    print(f"\n{pd.DataFrame(token_tags, columns=['Token', 'Tag']).to_markdown()}")


Processing dev data...


Processing dev data...: 100%|██████████| 108/108 [00:16<00:00,  6.48it/s]

Dev data processed: 108 sentences

Sample dev sentence:
Document ID: mh_549
  Text: The gut microbiota and mental health in adults.  A growing body of evidence point toward the bidirectional gut microbiota-brain axis playing a role in mental health. Most of this research is conducted on animals why we in this review summarize and comment upon recent studies evaluating the gut microbiome in mental health in humans. Further support for the relevance of the bidirectional gut microbiota-brain communication in mood disorders has been presented, such as the effect of probiotics on brain connectivity and mental health outcomes and pregnancy related stress on gut microbiota in the newborn child. However, the heterogeneity between studies precludes conclusions regarding differences in microbiota composition in mental disease and health and many of the studies are limited by a cross-sectional design, small sample sizes and multiple comparisons. Thus, well-designed longitudinal studies with large

## Prepare Dataset for BERT Training

In [29]:
class TokenClassificationDataset(Dataset):
    """Custom dataset for token classification with BERT."""
    
    def __init__(self, texts, tokens, labels, tokenizer, max_length=512):
        self.texts = texts
        self.tokens = tokens
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        original_labels = self.labels[idx]
        
        # Tokenize the text
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        input_ids = encoding['input_ids'].squeeze()
        attention_mask = encoding['attention_mask'].squeeze()
        
        # Initialize labels with -100 (ignored by loss)
        labels = torch.full((self.max_length,), -100, dtype=torch.long)
        
        # Align original labels with tokenized input
        for i, token_id in enumerate(input_ids):
            if i == 0:  # CLS token
                labels[i] = -100
            elif token_id == self.tokenizer.sep_token_id:  # SEP token
                labels[i] = -100
            elif token_id == self.tokenizer.pad_token_id:  # PAD token
                labels[i] = -100
            else:
                original_idx = i - 1
                if original_idx < len(original_labels):
                    labels[i] = original_labels[original_idx]
                else:
                    labels[i] = -100
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }


print("Custom dataset class defined")

Custom dataset class defined


In [30]:
# Create datasets
print("Creating training datasets...")

train_dataset = TokenClassificationDataset(
    texts=[entry['text'] for entry in train_data.values()],
    tokens=[entry['tokens'] for entry in train_data.values()],
    labels=[entry['ner_tags'] for entry in train_data.values()],
    tokenizer=tokenizer
)

dev_dataset = TokenClassificationDataset(
    texts=[entry['text'] for entry in dev_data.values()],
    tokens=[entry['tokens'] for entry in dev_data.values()],
    labels=[entry['ner_tags'] for entry in dev_data.values()],
    tokenizer=tokenizer
)

print(f"Training dataset: {len(train_dataset)} examples")
print(f"Dev dataset: {len(dev_dataset)} examples")

Creating training datasets...
Training dataset: 432 examples
Dev dataset: 108 examples


## Configure Training Arguments

In [31]:
# Setup data collator for token classification
# Data collator is used to dynamically pad inputs and labels
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt"
)
print("Data collator initialized")

Data collator initialized


In [32]:
# Define training arguments
training_args = TrainingArguments(
    output_dir=output_model_dir,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
    logging_steps=100,
    save_total_limit=2,
    seed=42,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

print("Training configuration ready")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")

Training configuration ready
  Batch size: 16
  Epochs: 3
  Learning rate: 2e-05


## Train BERT Model

Note: This cell might take several minutes to run.


**Additional configurations to test**
- Aggregate training samples per paragraph/document
- Change hyperparameters (*learning_rate*, *batch_size*, *num_train_epochs*, *weight_decay*)

In [ ]:
# Initialize Trainer
print("Initializing Trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

print("Trainer initialized")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Evaluation samples: {len(dev_dataset)}")

C:\Users\marti\AppData\Local\Temp\ipykernel_43116\1594551960.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Initializing Trainer...
✓ Trainer initialized
  Training samples: 432
  Evaluation samples: 108


In [34]:
# Start training
print("="*60)
print("Starting model training...")
print("="*60)

import time
training_start_time = time.time()

train_result = trainer.train()

training_duration = time.time() - training_start_time

print("\n" + "="*60)
print("✓ TRAINING COMPLETED!")
print("="*60)
print(f"Training time: {training_duration/60:.2f} minutes")

Starting model training...


Epoch,Training Loss,Validation Loss
1,No log,0.352720
2,No log,0.300158
3,No log,0.289211



✓ TRAINING COMPLETED!
Training time: 5.92 minutes


## Save Trained Model

In [37]:
# Save the trained model
print("Saving trained model...")

os.makedirs(output_model_dir, exist_ok=True)
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)

print(f"Model saved to: {output_model_dir}")

Saving trained model...
Model saved to: models/bert-base-uncased


## Inference Function

In [38]:
# Load the trained model for inference
print("Loading trained model for inference...")

inference_model = AutoModelForTokenClassification.from_pretrained(output_model_dir)
inference_tokenizer = AutoTokenizer.from_pretrained(output_model_dir)
inference_model.eval()

print(f"Model loaded from: {output_model_dir}")

Loading trained model for inference...
Model loaded from: models/bert-base-uncased


## Predict on Test Set

In [39]:
def perform_inference(model, tokenizer, text, id2label):
    """Perform token classification inference on a single text."""
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        return_offsets_mapping=True
    )
    
    with torch.no_grad():
        outputs = model(**{k: v for k, v in inputs.items() if k != 'offset_mapping'})
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_labels = torch.argmax(predictions, dim=-1)
    
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    labels = [id2label[pred.item()] for pred in predicted_labels[0]]
    
    # Extract terms using BIO scheme
    predicted_terms = []
    current_term = []
    
    for token, label in zip(tokens, labels):
        if token in ['[CLS]', '[SEP]', '[PAD]']:
            continue
            
        if label == 'B-TERM':
            if current_term:
                predicted_terms.append(tokenizer.convert_tokens_to_string(current_term))
            current_term = [token]
        elif label == 'I-TERM' and current_term:
            current_term.append(token)
        else:
            if current_term:
                predicted_terms.append(tokenizer.convert_tokens_to_string(current_term))
                current_term = []
    
    if current_term:
        predicted_terms.append(tokenizer.convert_tokens_to_string(current_term))
    
    # Clean predicted terms
    predicted_terms = [term.replace(' ##', '').strip() for term in predicted_terms if term.strip()]
    
    return predicted_terms


print("Inference function defined")

Inference function defined


In [46]:
from tqdm import tqdm

# Run inference on all test documents
test_data = json.load(open('../data/taskA/test.json', 'r', encoding='utf-8'))
print("Running inference on test set...")
bert_preds = []

for id, content in tqdm(test_data.items(), desc="Performing inference..."):
    predicted_terms = perform_inference(
        inference_model,
        inference_tokenizer,
        content['text'],
        id2label
    )
    bert_preds.append(predicted_terms)

# Match lowercase terms with original text to preserve casing
for i, (id, content) in enumerate(test_data.items()):
    original_text = content['text']
    original_terms = content.get('terms', [])
    predicted_terms = bert_preds[i]
    
    # Create a mapping of lowercase terms to their original casing
    term_mapping = {term.lower(): term for term in original_terms}
    
    # Replace predicted terms with their original casing if available
    bert_preds[i] = [term_mapping.get(term.lower(), term) for term in predicted_terms]

print(f"Inference completed: {len(bert_preds)} predictions")

Running inference on test set...


Performing inference...:   0%|          | 0/567 [00:00<?, ?it/s]

Performing inference...: 100%|██████████| 567/567 [03:53<00:00,  2.43it/s]

Inference completed: 567 predictions


In [47]:
# Save predictions in competition format
def save_predictions(predictions, documents, output_path):
    """Save predictions in competition format."""
    output = {}
    idx = 0
    for id, content in documents.items():
        output[id] = {
            'text': content['text'],
            'terms': predictions[idx]
        }
        idx += 1
    
    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(predictions)} predictions to {output_path}")

save_predictions(bert_preds, test_data, f'predictions/subtaskA_{model_name.split("/")[-1].lower()}_preds.json')

Saved 567 predictions to predictions/subtaskA_bert-base-uncased_preds.json


## Example Predictions

In [48]:
predictions = json.load(open(f'predictions/subtaskA_{model_name.split("/")[-1].lower()}_preds.json', 'r', encoding='utf-8'))

# Show example predictions
print("Example Predictions:\n")

count = 0

for id, content in test_data.items():
    if len(content['terms']) > 0 and count < 5:
        print(f"Document ID: {id}")
        print(f"Text: {content['text'][:100]}...")
        print(f"Gold terms: {content['terms'][:5]}")
        print(f"BERT predictions: {predictions[id]['terms'][:5]}")
        
        correct = set(content['terms']) & set(predictions[id]['terms'])
        missed = set(content['terms']) - set(predictions[id]['terms'])
        wrong = set(predictions[id]['terms']) - set(content['terms'])
        
        print(f"Correct: {len(correct)}")
        print(f"Missed: {len(missed)}")
        print(f"Wrong: {len(wrong)}")
        print("-"*80)
        print()
        
        count += 1


Example Predictions:

Document ID: mh_2
Text: Gut microbiota disturbances in hospitalized older adults with malnutrition and clinical outcomes.  O...
Gold terms: ['Gut microbiota disturbances', 'malnutrition', 'clinical outcomes', 'Malnutrition', 'gut microbiota']
BERT predictions: ['Gut microbiota disturbances', 'hospitalized', 'malnutrition', 'malnutrition', 'geriatric']
Correct: 13
Missed: 20
Wrong: 16
--------------------------------------------------------------------------------

Document ID: mh_3
Text: Preface: Special issue: "Ion channels and genetic epilepsy".  This preface introduces the Journal of...
Gold terms: ['Ion channels', 'genetic epilepsy', 'Epilepsy', 'disease', 'recurrent seizures']
BERT predictions: ['ion channels', 'genetic epilepsy', 'neurochemistry', 'epilepsy', 'epilepsy']
Correct: 16
Missed: 14
Wrong: 8
--------------------------------------------------------------------------------

Document ID: mh_5
Text: Association between gut microbiota dysbiosis and poo